# DataParallel 实现多 GPU 训练

这里多 GPU 训练使用的是数据并行方式

In [ ]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import torch
from torch import nn
from torch.optim import Adam

获取训练数据和验证数据

这里用的是手写数字识别的数据集

In [ ]:
train_dataset=MNIST(root='./data',download=True,transform=ToTensor(),train=True)
val_dataset=MNIST(root='./data',download=True,transform=ToTensor(),train=False)


train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True,
    persistent_workers=True,
    num_workers=4
)


val_data_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True,
    persistent_workers=True,
    num_workers=4
)

设计一个测试的分类模型

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(28 * 28, 1024)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(1024, 1024)
        self.relu2 = nn.ReLU()
        self.linear3 = nn.Linear(1024, 10)

    def forward(self, x):
        return self.linear3(self.relu2(self.linear2(self.relu1(self.linear1(self.flatten(x))))))

选择使用哪些编号的 GPU

并设定主 GPU

In [ ]:
device_ids = [0, 1, 2, 3, 4, 5, 6, 7]
main_device = torch.device(f'cuda:{device_ids[0]}')

将模型放到主 GPU 上

In [ ]:
model = Model().to(main_device)

使用 DataParallel 将模型放到多个 GPU 上

In [ ]:
model = nn.DataParallel(model, device_ids)

创建优化器

In [ ]:
optimizer = Adam(model.parameters())

开始训练

每一个 epoch 进行一次验证

分别输出每个 epoch 中训练过程和验证过程的损失总和

In [ ]:
for epoch in range(30):
    model.train()
    train_loss = 0
    for x, label in train_data_loader:
        optimizer.zero_grad()
        x, label = x.to(main_device), label.to(main_device)
        logits = model(x)
        loss = nn.functional.cross_entropy(logits, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, label in val_data_loader:
            x, label = x.to(main_device), label.to(main_device)
            logits = model(x)
            loss = nn.functional.cross_entropy(logits, label)
            val_loss += loss.item()
    print(f'train epoch: {epoch}, train loss: {train_loss}, val loss: {val_loss}')

保存最终得到的模型

In [ ]:
torch.save(model.module.state_dict(), 'model.pth')